In [0]:
import polars as pl

df = pl.scan_csv("sample_orders.csv")

country_map = {
    "Poland": "PL",
    "Germany": "DE",
    "United Kingdom": "UK",
    "Spain": "ES",
    "France": "FR",
    "Netherlands": "NL",
    "Ireland": "IE",
    "Sweden": "SE",
    "Italy": "IT",
    "Portugal": "PT"
}

df = (
    df.with_columns([
    pl.col("country")
    .replace_strict(country_map, default="other")
    .alias("country"),

    (pl.col("price_each") * pl.col("quantity")).alias("revenue"),

    pl.col("placed_at")
      .str.strptime(pl.Datetime, "%Y-%m-%d", strict=False)
      .alias("placed_at")
])
.filter(pl.col("status") == "paid")
.group_by("country")
.agg(
        pl.col("revenue").sum().alias("revenue"),
        pl.col("id").count().alias("orders")
    )
.sort("revenue", descending=True)
)

result = df.collect()
display(result)



